###   Лабараторная работа 6

####  Сколько я смогу получить


In [1]:
import pandas as pd
import numpy as np
import math

import matplotlib
import matplotlib.pyplot as plt
import random


%matplotlib inline
import os
os.chdir("C:/Users/olesh/data_analys/GI")
exp45=207800

In [25]:
excel_data_df = pd.read_excel('опрос ззз.xlsx', sheet_name = 'Копии', index_col=False)
excel_data_df

,Время,Ответ
0,2025-12-11 22:05:21.130,"Янаги, Эвелин, Солдат 0 Энби, Исюань, Укинами ..."
1,2025-12-11 21:29:05.230,"Янаги, Эвелин, Сид, Йидхари Мёрфи"
2,2025-12-11 21:57:48.009,"Янаги, Эвелин, Пань Иньху, Укинами Юдзуха, Йид..."
3,2025-12-11 21:27:15.773,"Янаги, Эвелин, Йидхари Мёрфи"
4,2025-12-11 21:46:43.255,"Янаги, Эвелин, Гашетка, Сид, Баньюэ"
...,...,...
4465,2025-12-11 21:37:54.804,NaN
4466,2025-12-11 21:47:12.595,NaN
4467,2025-12-11 22:01:53.719,NaN
4468,2025-12-11 22:12:09.843,NaN


In [26]:
# --- 1. Удаляем пустые голоса ---
excel_data_df = excel_data_df.dropna()
# --- 2. Оставляем строки с 1 или 2 персонажами ---
# считаем запятые
excel_data_df["comma_count"] = excel_data_df["Ответ"].str.count(",")
# нам нужны строки с 0 или 1 запятой
excel_data_df = excel_data_df[excel_data_df["comma_count"] <= 1]
# лишняя колонка больше не нужна
excel_data_df = excel_data_df.drop(columns=["comma_count"])
# разделяем колонки по 1 персонажу
excel_data_df["chars"] = excel_data_df["Ответ"].str.split(", ")
excel_data_df = excel_data_df.explode("chars")
excel_data_df = excel_data_df.drop(columns=["Ответ"])
# убеждаемся, что timestamp — datetime
excel_data_df["Время"] = pd.to_datetime(excel_data_df["Время"])

<ipython-input-26-586534dbfecc>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  excel_data_df["comma_count"] = excel_data_df["Ответ"].str.count(",")


In [28]:
# --- 3. Смотрим групы одинаковых голосов ---
# сортируем, чтобы разница во времени считалась подряд
excel_data_df = excel_data_df.sort_values("Время")
# считаем разницу во времени в каждой группе одинаковых голосов
excel_data_df["time_diff"] = excel_data_df.groupby("chars")["Время"].diff()

In [29]:
# --- 4. Оставляем только строки, где разница < 1 минуты ---
result = excel_data_df[excel_data_df["time_diff"] < pd.Timedelta(minutes=1)]
result

In [ ]:
series = result.groupby("chars").size()
print(series)

In [39]:
series

chars
Антон               1
Астра Яо            3
Баньюэ             37
Билли               1
Бёрнис              3
Вивиан             12
Гашетка            11
Джейн              28
Диалинь            17
Исюань             22
Йидхари Мёрфи       6
Комано Монато      22
Лайтер             22
Ликаон             27
Люси                1
Мияби              66
Николь              1
Орфея и Магус      22
Пайпер              4
Пульхра             1
Сид               104
Сокаку              1
Солдат 0 Энби       2
Укинами Юдзуха     21
Харумаса            5
Хуго Влад          20
Цезарь              2
Цзюй Фуфу           4
Цинъи               1
Чжу Юань            2
Эвелин              9
Элис Таймфилд       8
Эллен              11
Энби                5
Янаги               2
dtype: int64

In [2]:
excel_data_df = pd.read_excel('конкурс артов.xlsx', sheet_name = 'Мертвая мать 2.4', index_col=False)
name_to_delete = excel_data_df[(excel_data_df.сумма < 600) & (excel_data_df['число артов'] == 4)].участник
name_to_delete = []
for name in name_to_delete:
    excel_data_df = excel_data_df.drop(excel_data_df[excel_data_df.участник == name].index)
excel_data_df = excel_data_df.drop(columns=['ранг', 'опыт на ранге'])
excel_data_df

,участник,число артов,цветок,перо,часы,кубок,шапка,сумма,опыт
0,олежа#1488,1,39.6,31.0,27.2,42.0,35.7,175.5,790766
1,олежа#1488,4,143.8,109.4,102.6,115.1,78.6,549.5,790766
2,astrogen1x#3933,1,41.2,37.2,45.8,36.5,24.9,185.6,722938
3,astrogen1x#3933,4,153.3,132.8,162.4,117.3,87.1,652.9,722938
4,GoodJuJu#2806,1,42.8,35.0,39.6,41.2,27.2,185.8,976788
...,...,...,...,...,...,...,...,...,...
321,kimikoto#2973,4,148.8,148.8,148.8,148.8,74.4,669.4,1143856
322,kimikoto#2973 твин,1,35.1,35.1,35.1,35.1,17.6,158.1,809778
323,kimikoto#2973 твин,4,140.5,140.5,140.5,140.5,70.2,632.2,809778
324,Prohor24#9124,1,25.8,25.8,25.8,25.8,12.9,116.0,542004


In [3]:
#excel_data_df = pd.read_excel('игра ивенты.xlsx', sheet_name = 'Мертвая мать 2.4', index_col=False)
#excel_data_df = excel_data_df.drop(excel_data_df[excel_data_df.участник == 'GL1n#3501 твин'].index)

one_best_frame = excel_data_df[excel_data_df['число артов'] == 1].drop(columns=['число артов'])
four_best_frame = excel_data_df[excel_data_df['число артов'] == 4].drop(columns=['число артов'])

In [4]:
#excel_data_df.loc[excel_data_df.участник == 'GL1n#3501 мейн', 'опыт на ранге']=100000000
#excel_data_df


In [5]:
one_best_frame['ранговый коэффициент'] = one_best_frame['опыт']-exp45
one_best_frame['ранговый коэффициент'] =np.log10(one_best_frame['ранговый коэффициент'])
one_best_frame['ранговый коэффициент'] = one_best_frame['ранговый коэффициент']/one_best_frame['ранговый коэффициент'].max()
one_best_frame = one_best_frame.drop(columns=['опыт'])
one_best_frame['цветок b'] = np.around(one_best_frame['цветок'] / one_best_frame['ранговый коэффициент'], 1)
one_best_frame['перо b'] = np.around(one_best_frame['перо'] / one_best_frame['ранговый коэффициент'], 1)
one_best_frame['часы b'] = np.around(one_best_frame['часы'] / one_best_frame['ранговый коэффициент'], 1)
one_best_frame['кубок b'] = np.around(one_best_frame['кубок'] / one_best_frame['ранговый коэффициент'], 1)
one_best_frame['шапка b'] = np.around(one_best_frame['шапка'] / one_best_frame['ранговый коэффициент'], 1)
one_best_frame['сумма b'] = np.around(one_best_frame['сумма'] / one_best_frame['ранговый коэффициент'], 1)

In [6]:
one_best_frame

,участник,цветок,перо,часы,кубок,шапка,сумма,ранговый коэффициент,цветок b,перо b,часы b,кубок b,шапка b,сумма b
0,олежа#1488,39.6,31.0,27.2,42.0,35.7,175.5,0.901069,43.9,34.4,30.2,46.6,39.6,194.8
2,astrogen1x#3933,41.2,37.2,45.8,36.5,24.9,185.6,0.892673,46.2,41.7,51.3,40.9,27.9,207.9
4,GoodJuJu#2806,42.8,35.0,39.6,41.2,27.2,185.8,0.919866,46.5,38.0,43.0,44.8,29.6,202.0
6,True slave#5556,42.8,38.0,36.5,37.3,28.0,182.6,0.915030,46.8,41.5,39.9,40.8,30.6,199.6
8,aira#2442,37.4,41.8,35.7,40.4,28.8,184.1,0.932893,40.1,44.8,38.3,43.3,30.9,197.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316,ARRA#8194,45.0,40.4,46.7,47.4,35.7,215.2,0.936003,48.1,43.2,49.9,50.6,38.1,229.9
318,newerar#4093,48.2,44.3,45.1,44.3,27.2,209.1,0.931699,51.7,47.5,48.4,47.5,29.2,224.4
320,kimikoto#2973,37.2,37.2,37.2,37.2,18.6,167.4,0.933209,39.9,39.9,39.9,39.9,19.9,179.4
322,kimikoto#2973 твин,35.1,35.1,35.1,35.1,17.6,158.1,0.903247,38.9,38.9,38.9,38.9,19.5,175.0


In [7]:
four_best_frame['ранговый коэффициент'] = four_best_frame['опыт']-exp45
four_best_frame['ранговый коэффициент'] =np.log10(four_best_frame['ранговый коэффициент'])
four_best_frame['ранговый коэффициент'] = four_best_frame['ранговый коэффициент']/four_best_frame['ранговый коэффициент'].max()
four_best_frame = four_best_frame.drop(columns=['опыт'])
four_best_frame['цветок b'] = np.around(four_best_frame['цветок'] / four_best_frame['ранговый коэффициент'], 1)
four_best_frame['перо b'] = np.around(four_best_frame['перо'] / four_best_frame['ранговый коэффициент'], 1)
four_best_frame['часы b'] = np.around(four_best_frame['часы'] / four_best_frame['ранговый коэффициент'], 1)
four_best_frame['кубок b'] = np.around(four_best_frame['кубок'] / four_best_frame['ранговый коэффициент'], 1)
four_best_frame['шапка b'] = np.around(four_best_frame['шапка'] / four_best_frame['ранговый коэффициент'], 1)
four_best_frame['сумма b'] = np.around(four_best_frame['сумма'] / four_best_frame['ранговый коэффициент'], 1)

In [8]:
four_best_frame

,участник,цветок,перо,часы,кубок,шапка,сумма,ранговый коэффициент,цветок b,перо b,часы b,кубок b,шапка b,сумма b
1,олежа#1488,143.8,109.4,102.6,115.1,78.6,549.5,0.901069,159.6,121.4,113.9,127.7,87.2,609.8
3,astrogen1x#3933,153.3,132.8,162.4,117.3,87.1,652.9,0.892673,171.7,148.8,181.9,131.4,97.6,731.4
5,GoodJuJu#2806,152.4,133.6,139.8,125.1,85.5,636.4,0.919866,165.7,145.2,152.0,136.0,92.9,691.8
7,True slave#5556,151.5,137.8,126.7,119.7,80.1,615.8,0.915030,165.6,150.6,138.5,130.8,87.5,673.0
9,aira#2442,141.0,150.6,119.6,146.9,104.9,663.0,0.932893,151.1,161.4,128.2,157.5,112.4,710.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,ARRA#8194,166.2,159.4,164.1,158.6,124.3,772.6,0.936003,177.6,170.3,175.3,169.4,132.8,825.4
319,newerar#4093,178.8,173.4,161.8,150.0,101.8,765.8,0.931699,191.9,186.1,173.7,161.0,109.3,821.9
321,kimikoto#2973,148.8,148.8,148.8,148.8,74.4,669.4,0.933209,159.4,159.4,159.4,159.4,79.7,717.3
323,kimikoto#2973 твин,140.5,140.5,140.5,140.5,70.2,632.2,0.903247,155.5,155.5,155.5,155.5,77.7,699.9


In [9]:
points_frame = pd.DataFrame(four_best_frame['участник'])
points_frame.set_index('участник', inplace=True)
points_frame['points'] = 0


# points_frame.loc['Юджи#4565 мейн']-=9
# points_frame.loc['Юджи#4565 мейн']-=1
# one_best_frame['участник'].iloc[-3]
points_frame

,points
участник,
олежа#1488,0
astrogen1x#3933,0
GoodJuJu#2806,0
True slave#5556,0
aira#2442,0
...,...
ARRA#8194,0
newerar#4093,0
kimikoto#2973,0


In [10]:
list_type = ['цветок', 'перо', 'часы', 'кубок', 'шапка',
       'цветок b', 'перо b', 'часы b', 'кубок b', 'шапка b']
list_sum = ['сумма', 'сумма b']

In [11]:
def count_points_by_frame(frame, coef = 1):
    for art_type in list_type:
        frame = frame.sort_values(by=[art_type])
        #points_frame.loc[frame['участник'].iloc[0]] -=3 * coef
        #points_frame.loc[frame['участник'].iloc[1]] -=2 * coef
        #points_frame.loc[frame['участник'].iloc[2]] -=1 * coef
        points_frame.loc[frame['участник'].iloc[-1]] +=3 * coef
        points_frame.loc[frame['участник'].iloc[-2]] +=2 * coef
        points_frame.loc[frame['участник'].iloc[-3]] +=1 * coef
        
    for art_type in list_sum:
        frame = frame.sort_values(by=[art_type])
        #points_frame.loc[frame['участник'].iloc[0]] -=6 * coef
        #points_frame.loc[frame['участник'].iloc[1]] -=4 * coef
        #points_frame.loc[frame['участник'].iloc[2]] -=2 * coef
        points_frame.loc[frame['участник'].iloc[-1]] +=6 * coef
        points_frame.loc[frame['участник'].iloc[-2]] +=4 * coef
        points_frame.loc[frame['участник'].iloc[-3]] +=2 * coef
        

In [12]:
#count_points_by_frame(one_best_frame, 0.5)
#count_points_by_frame(four_best_frame)
# points_frame.sort_values(by=['points'], ascending=False)

In [13]:
points_frame['points']=0
users = four_best_frame['участник']
for user in users:
    points = one_best_frame[one_best_frame.участник == user]['сумма'].iloc[0] * 0.1
    points += one_best_frame[one_best_frame.участник == user]['сумма b'].iloc[0] * 0.1 / 2
    points += four_best_frame[four_best_frame.участник == user]['сумма'].iloc[0]
    points += four_best_frame[four_best_frame.участник == user]['сумма b'].iloc[0] / 2
    points_frame.loc[user] += points
    

In [14]:
points_frame.sort_values(by=['points'], ascending=False)

,points
участник,
kimani#3989,1244.635
V_Vlad#5348,1233.185
Nogard#8988,1232.155
Bell⚔️#5796,1225.710
ARRA#8194,1218.315
...,...
和葉#5937,699.685
ohimadivergent#6813,686.960
t0b1x#3822,681.770


In [15]:
points_frame.sort_values(by=['points'], ascending=False).head(10)

,points
участник,
kimani#3989,1244.635
V_Vlad#5348,1233.185
Nogard#8988,1232.155
Bell⚔️#5796,1225.710
ARRA#8194,1218.315
newerar#4093,1208.880
Зеленый (Dummy_unit)#7917,1208.260
OKSID#1788,1198.315
f1llz#2298,1190.075


In [16]:
excel_data_df = pd.read_excel('конкурс артов.xlsx', sheet_name = 'Мертвая мать 2.4', index_col=False)
excel_data_df = excel_data_df[excel_data_df['число артов'] == 4].drop(columns=['число артов',
                                                                               'цветок', 'перо', 'часы', 'кубок', 'шапка',
                                                                              'опыт на ранге', 'опыт'])
top_table = excel_data_df.join(points_frame, on='участник').sort_values(by=['points'], ascending=False)
top_table

,участник,сумма,ранг,points
149,kimani#3989,789.7,57,1244.635
245,V_Vlad#5348,786.4,58,1233.185
93,Nogard#8988,786.1,58,1232.155
119,Bell⚔️#5796,780.5,58,1225.710
317,ARRA#8194,772.6,57,1218.315
...,...,...,...,...
181,和葉#5937,430.8,55,699.685
255,ohimadivergent#6813,425.9,56,686.960
193,t0b1x#3822,427.0,56,681.770
19,slowlilac#0626,333.4,55,543.210


In [17]:
top_table.head(10)

,участник,сумма,ранг,points
149,kimani#3989,789.7,57,1244.635
245,V_Vlad#5348,786.4,58,1233.185
93,Nogard#8988,786.1,58,1232.155
119,Bell⚔️#5796,780.5,58,1225.710
317,ARRA#8194,772.6,57,1218.315
319,newerar#4093,765.8,57,1208.880
235,Зеленый (Dummy_unit)#7917,766.3,57,1208.260
195,OKSID#1788,767.1,59,1198.315
169,f1llz#2298,759.6,59,1190.075
301,Polly#8913,757.0,58,1185.050


In [18]:
ind = 1
for i, row in top_table.iterrows():
    if row.участник == 'Asura#1488' or row.участник == 'ARRA#8194' or row.участник == 'newerar#4093':
        continue
    print(f"{ind}. @{row.участник} - {row.ранг} ранг, {(row.сумма+248.8):0.1f} критмассы в артефактах")
    ind += 1
    if ind == 11:
        break

1. @kimani#3989 - 57 ранг, 1038.5 критмассы в артефактах
2. @V_Vlad#5348 - 58 ранг, 1035.2 критмассы в артефактах
3. @Nogard#8988 - 58 ранг, 1034.9 критмассы в артефактах
4. @Bell⚔️#5796 - 58 ранг, 1029.3 критмассы в артефактах
5. @Зеленый (Dummy_unit)#7917 - 57 ранг, 1015.1 критмассы в артефактах
6. @OKSID#1788 - 59 ранг, 1015.9 критмассы в артефактах
7. @f1llz#2298 - 59 ранг, 1008.4 критмассы в артефактах
8. @Polly#8913 - 58 ранг, 1005.8 критмассы в артефактах
9. @Modeus#2990 - 58 ранг, 997.1 критмассы в артефактах
10. @JR Vlooo#6896 - 57 ранг, 991.7 критмассы в артефактах


In [19]:
top_table.to_excel('result.xlsx')
top_table

,участник,сумма,ранг,points
149,kimani#3989,789.7,57,1244.635
245,V_Vlad#5348,786.4,58,1233.185
93,Nogard#8988,786.1,58,1232.155
119,Bell⚔️#5796,780.5,58,1225.710
317,ARRA#8194,772.6,57,1218.315
...,...,...,...,...
181,和葉#5937,430.8,55,699.685
255,ohimadivergent#6813,425.9,56,686.960
193,t0b1x#3822,427.0,56,681.770
19,slowlilac#0626,333.4,55,543.210
